In [3]:
# ============================================================
# SNIPPET 1 / 5 — SETUP, DATA LOADING & EXPLORATORY ANALYSIS
# Cardiovascular Disease Classifier | Portfolio Project
# Dataset: sulianova/cardiovascular-disease-dataset (Kaggle)
# ============================================================
# KAGGLE SETUP: Add the dataset first:
#   Notebook Settings → Add Data → search "cardiovascular-disease-dataset" by sulianova
#   The CSV path will be: /kaggle/input/cardiovascular-disease-dataset/cardio_train.csv
# ============================================================

# ── 1. INSTALLS (Kaggle has most; this ensures SHAP is present) ──────────────
import subprocess
subprocess.run(["pip", "install", "shap", "-q"])

# ── 2. CORE IMPORTS ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Set a clean, portfolio-friendly plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
PALETTE = {"0": "#4C72B0", "1": "#DD8452"}   # reuse this everywhere for consistency

# ── 3. LOAD DATA ─────────────────────────────────────────────────────────────
PATH = "/kaggle/input/datasets/sulianova/cardiovascular-disease-dataset/cardio_train.csv"
df = pd.read_csv(PATH, sep=";")   # NOTE: this dataset uses semicolon delimiter

print(f"Shape: {df.shape}")       # expect (70000, 13)
print(df.dtypes)
df.head(3)

# ── 4. BASIC SANITY CHECKS ───────────────────────────────────────────────────
print("\n── Null counts ──")
print(df.isnull().sum())          # this dataset is pre-cleaned; expect all zeros

print("\n── Target distribution ──")
print(df["cardio"].value_counts(normalize=True).round(3))
# Expect ~50/50 — the dataset is deliberately balanced, so no SMOTE needed here,
# but we will still use class_weight="balanced" in models as a defensive measure.

# ── 5. FEATURE ENGINEERING ───────────────────────────────────────────────────
# age is stored in DAYS — convert to years (much more interpretable in plots & SHAP)
df["age_years"] = (df["age"] / 365.25).round(1)

# BMI from height (cm) and weight (kg) — a standard clinical risk factor
# BMI = weight(kg) / height(m)^2
df["bmi"] = (df["weight"] / (df["height"] / 100) ** 2).round(2)

# Pulse pressure = systolic - diastolic (marker of arterial stiffness)
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]

# ── 6. REMOVE PHYSIOLOGICALLY IMPOSSIBLE VALUES ──────────────────────────────
# Blood pressure values outside survivable range are measurement errors.
# Evidence-based thresholds from clinical literature:
#   Systolic:  60–250 mmHg  (below 60 = shock; above 250 = extreme hypertensive crisis)
#   Diastolic: 40–150 mmHg
#   Height:    100–220 cm   (filter dwarfism data entry errors)
#   Weight:    30–200 kg
before = len(df)
df = df[
    (df["ap_hi"].between(60, 250)) &
    (df["ap_lo"].between(40, 150)) &
    (df["ap_hi"] > df["ap_lo"]) &   # systolic must exceed diastolic
    (df["height"].between(100, 220)) &
    (df["weight"].between(30, 200))
]
print(f"\nRemoved {before - len(df)} outlier rows ({(before-len(df))/before*100:.1f}%)")
print(f"Clean dataset size: {len(df)}")

# ── 7. EDA — FIGURE 1: DISTRIBUTION OF KEY FEATURES BY CLASS ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Feature distributions by cardiovascular disease status", fontsize=14, y=1.01)

features_to_plot = ["age_years", "bmi", "ap_hi", "ap_lo", "pulse_pressure", "cholesterol"]
labels = ["Age (years)", "BMI", "Systolic BP (mmHg)", "Diastolic BP (mmHg)",
          "Pulse pressure", "Cholesterol level"]

for ax, feat, label in zip(axes.flat, features_to_plot, labels):
    for val, color in zip([0, 1], ["#4C72B0", "#DD8452"]):
        subset = df[df["cardio"] == val][feat]
        ax.hist(subset, bins=40, alpha=0.55, color=color,
                label=f"{'No CVD' if val==0 else 'CVD'}", density=True)
    ax.set_xlabel(label)
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("eda_distributions.png", bbox_inches="tight")
plt.show()
print("Saved → eda_distributions.png")

# ── 8. EDA — FIGURE 2: CATEGORICAL FEATURE PREVALENCE ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Categorical risk factor prevalence by CVD status", fontsize=13)

cat_features = ["cholesterol", "gluc", "smoke"]
cat_labels   = ["Cholesterol (1=normal, 2=above, 3=well above)",
                "Glucose (1=normal, 2=above, 3=well above)",
                "Smoking status"]

for ax, feat, label in zip(axes, cat_features, cat_labels):
    # Compute CVD rate per category value
    rates = df.groupby(feat)["cardio"].mean().reset_index()
    counts = df[feat].value_counts().sort_index()
    bars = ax.bar(rates[feat].astype(str), rates["cardio"],
                  color="#4C72B0", alpha=0.8, edgecolor="white")
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("CVD rate")
    ax.set_ylim(0, 0.8)
    # Annotate bars with sample counts
    for bar, (cat, n) in zip(bars, counts.items()):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f"n={n:,}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("eda_categorical.png", bbox_inches="tight")
plt.show()
print("Saved → eda_categorical.png")

# ── 9. EDA — FIGURE 3: CORRELATION HEATMAP ───────────────────────────────────
num_cols = ["age_years", "height", "weight", "bmi", "ap_hi", "ap_lo",
            "pulse_pressure", "cholesterol", "gluc", "cardio"]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))   # show only lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.4,
            annot_kws={"size": 9}, ax=ax)
ax.set_title("Pearson correlation — numerical features", fontsize=13)
plt.tight_layout()
plt.savefig("eda_correlation.png", bbox_inches="tight")
plt.show()
print("Saved → eda_correlation.png")

# ── 10. EDA — FIGURE 4: AGE vs BP SCATTER (coloured by target) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Age vs blood pressure by CVD status", fontsize=13)

# Sample 5000 points to keep the scatter readable (full 70k is a blob)
sample = df.sample(5000, random_state=42)

for ax, bp_col, label in zip(axes,
                              ["ap_hi", "ap_lo"],
                              ["Systolic BP", "Diastolic BP"]):
    for val, color, name in zip([0, 1], ["#4C72B0", "#DD8452"], ["No CVD", "CVD"]):
        sub = sample[sample["cardio"] == val]
        ax.scatter(sub["age_years"], sub[bp_col],
                   alpha=0.25, s=10, c=color, label=name)
    ax.set_xlabel("Age (years)")
    ax.set_ylabel(f"{label} (mmHg)")
    ax.legend(fontsize=9, markerscale=2)

plt.tight_layout()
plt.savefig("eda_scatter.png", bbox_inches="tight")
plt.show()
print("Saved → eda_scatter.png")

# ── 11. PRINT SUMMARY STATS FOR REVIEW ───────────────────────────────────────
print("\n── Engineered feature summary ──")
print(df[["age_years", "bmi", "pulse_pressure"]].describe().round(2))

print("\n── Class balance after cleaning ──")
vc = df["cardio"].value_counts()
print(f"  No CVD : {vc[0]:,}  ({vc[0]/len(df)*100:.1f}%)")
print(f"  CVD    : {vc[1]:,}  ({vc[1]/len(df)*100:.1f}%)")

print("\n✅ Snippet 1 complete.")
print("   Outputs: eda_distributions.png, eda_categorical.png,")
print("            eda_correlation.png, eda_scatter.png")
print("   Proceed to Snippet 2 → preprocessing pipeline + feature selection.")

Shape: (70000, 13)
id               int64
age              int64
gender           int64
height           int64
weight         float64
ap_hi            int64
ap_lo            int64
cholesterol      int64
gluc             int64
smoke            int64
alco             int64
active           int64
cardio           int64
dtype: object

── Null counts ──
id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64

── Target distribution ──
cardio
0    0.5
1    0.5
Name: proportion, dtype: float64

Removed 1366 outlier rows (2.0%)
Clean dataset size: 68634
Saved → eda_distributions.png
Saved → eda_categorical.png
Saved → eda_correlation.png
Saved → eda_scatter.png

── Engineered feature summary ──
       age_years       bmi  pulse_pressure
count   68634.00  68634.00        68634.00
mean       53.29     27.47           45.

In [4]:
# ============================================================
# SNIPPET 2 / 5 — PREPROCESSING PIPELINE + FEATURE SELECTION
# Cardiovascular Disease Classifier | Portfolio Project
# ============================================================
# Builds on Snippet 1. Run in the SAME Kaggle notebook session.
# df must already be in memory (cleaned, with engineered features).
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# ── 1. FEATURE SELECTION — DROP REDUNDANT COLUMNS ────────────────────────────
# Reasoning from correlation heatmap (Snippet 1):
#   • weight (r=0.85 with bmi) and height (r=-0.22 with bmi) are redundant
#     now that we have bmi. Dropping them removes near-collinearity.
#   • age (days) is fully replaced by age_years — drop original.
#   • id column carries no signal.

COLS_TO_DROP = ["id", "age", "weight", "height"]
df_model = df.drop(columns=COLS_TO_DROP)

print("── Columns kept for modeling ──")
print(df_model.columns.tolist())

# ── 2. BMI CAP — fix the outlier (max=152.55 spotted in Snippet 1) ───────────
# Clinical ceiling: BMI > 60 is extreme morbid obesity AND likely a data entry
# error (e.g., height/weight transposed). Cap rather than drop to keep samples.
bmi_cap = 60
n_capped = (df_model["bmi"] > bmi_cap).sum()
df_model["bmi"] = df_model["bmi"].clip(upper=bmi_cap)
print(f"\nBMI capped at {bmi_cap}: {n_capped} rows adjusted")

# ── 3. DEFINE FEATURE GROUPS ─────────────────────────────────────────────────
# Continuous features → StandardScaler (needed for LR; harmless for trees)
# Binary/ordinal features → pass through as-is (already 0/1 or 1/2/3)

CONTINUOUS = ["age_years", "bmi", "ap_hi", "ap_lo", "pulse_pressure"]
BINARY     = ["gender", "smoke", "alco", "active"]
ORDINAL    = ["cholesterol", "gluc"]   # 1=normal, 2=above normal, 3=well above
TARGET     = "cardio"

# Sanity check — make sure all columns are accounted for
expected = set(CONTINUOUS + BINARY + ORDINAL + [TARGET])
actual   = set(df_model.columns)
assert expected == actual, f"Column mismatch: {expected.symmetric_difference(actual)}"
print(f"\nFeature groups confirmed — {len(CONTINUOUS)} continuous, "
      f"{len(BINARY)} binary, {len(ORDINAL)} ordinal")

# ── 4. TRAIN / TEST SPLIT ─────────────────────────────────────────────────────
# • Stratified to preserve 50/50 class balance in both splits
# • 80/20 split: with ~68k rows this gives ~54.9k train / ~13.7k test
#   At 5-fold CV on train: each fold = ~11k validation samples — statistically
#   robust (Central Limit Theorem kicks in well above n=1000 per class)
# • random_state=42 for full reproducibility

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"\nTrain size : {len(X_train):,}  "
      f"(CVD={y_train.sum():,}, No CVD={( y_train==0).sum():,})")
print(f"Test size  : {len(X_test):,}  "
      f"(CVD={y_test.sum():,},  No CVD={(y_test==0).sum():,})")

# ── 5. PREPROCESSING PIPELINE ────────────────────────────────────────────────
# ColumnTransformer applies different transforms to different column groups.
# StandardScaler on continuous: zero-mean, unit-variance
#   → essential for Logistic Regression convergence & fair coefficient comparison
#   → harmless for XGBoost/LightGBM (they're scale-invariant, but consistent
#      pipelines prevent silent bugs when stacking models later)
# Binary and ordinal columns: passthrough (already numeric, meaningful scale)

preprocessor = ColumnTransformer(
    transformers=[
        ("scale",       StandardScaler(), CONTINUOUS),
        ("passthrough", "passthrough",    BINARY + ORDINAL),
    ],
    remainder="drop"   # safety net — drop anything not explicitly listed
)

# Column order after transform (needed to rebuild DataFrames for SHAP later)
FEATURE_NAMES_OUT = CONTINUOUS + BINARY + ORDINAL

# ── 6. FIT PREPROCESSOR ON TRAIN ONLY ────────────────────────────────────────
# Critical: fit on train, transform both. Never fit on test — data leakage.
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

X_train_df = pd.DataFrame(X_train_proc, columns=FEATURE_NAMES_OUT)
X_test_df  = pd.DataFrame(X_test_proc,  columns=FEATURE_NAMES_OUT)

print("\nPreprocessor fitted. Sample of scaled train data:")
print(X_train_df.head(3).round(3))

# ── 7. MUTUAL INFORMATION — FEATURE IMPORTANCE (MODEL-AGNOSTIC) ───────────────
# Mutual Information measures how much knowing a feature reduces uncertainty
# about the target. Unlike correlation, it captures non-linear relationships.
# We compute it on the RAW (unscaled) X_train because MI is rank-based
# and scaling does not change MI scores.

mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({
    "feature": X_train.columns,
    "mi_score": mi_scores
}).sort_values("mi_score", ascending=False)

print("\n── Mutual Information scores ──")
print(mi_df.to_string(index=False))

# ── 8. PLOT: MUTUAL INFORMATION ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#378ADD" if s > 0.02 else "#AAAAAA" for s in mi_df["mi_score"]]
ax.barh(mi_df["feature"], mi_df["mi_score"], color=colors, edgecolor="white")
ax.axvline(0.02, color="red", linestyle="--", linewidth=1,
           label="Threshold (0.02) — grey = weak")
ax.set_xlabel("Mutual Information score")
ax.set_title("Feature relevance — Mutual Information vs cardio target")
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("feature_importance_mi.png", bbox_inches="tight")
plt.show()
print("Saved → feature_importance_mi.png")

# ── 9. PERMUTATION IMPORTANCE (via quick LR baseline) ────────────────────────
# A second, model-based feature importance check using Logistic Regression.
# Permutation importance: shuffle one feature at a time → measure AUC drop.
# If shuffling a feature doesn't hurt performance, the model doesn't need it.
# We use LR here (fast) just to get importance ranks — actual models come in S3.

from sklearn.metrics import roc_auc_score

lr_quick = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr_quick.fit(X_train_proc, y_train)
baseline_auc = roc_auc_score(y_test, lr_quick.predict_proba(X_test_proc)[:, 1])
print(f"\nQuick LR baseline AUC (C=0.1): {baseline_auc:.4f}")

perm = permutation_importance(
    lr_quick, X_test_proc, y_test,
    n_repeats=10, scoring="roc_auc", random_state=42, n_jobs=-1
)
perm_df = pd.DataFrame({
    "feature":    FEATURE_NAMES_OUT,
    "importance": perm.importances_mean,
    "std":        perm.importances_std
}).sort_values("importance", ascending=False)

print("\n── Permutation Importance (LR, AUC drop) ──")
print(perm_df.to_string(index=False))

# ── 10. PLOT: PERMUTATION IMPORTANCE ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#DD8452" if v > 0 else "#BBBBBB" for v in perm_df["importance"]]
ax.barh(perm_df["feature"], perm_df["importance"],
        xerr=perm_df["std"], color=colors, edgecolor="white",
        error_kw={"elinewidth": 1, "capsize": 3})
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean AUC decrease when feature is shuffled")
ax.set_title("Permutation importance — Logistic Regression baseline")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("feature_importance_perm.png", bbox_inches="tight")
plt.show()
print("Saved → feature_importance_perm.png")

# ── 11. CROSS-VALIDATION SETUP (shared across Snippets 3 & 4) ────────────────
# Define the CV splitter here so Snippet 3 imports nothing extra.
# StratifiedKFold preserves class ratio in every fold.
# n_splits=5: industry standard for tabular medical data at this size.
#   • 5-fold gives 80/20 within train → each val fold ≈ 11k rows
#   • 10-fold would give tighter estimates but costs 2x compute
#   • 3-fold risks high variance estimates — too few val samples per fold
# shuffle=True + random_state for reproducibility across runs.

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── 12. SAVE ARTIFACTS FOR SNIPPET 3 ─────────────────────────────────────────
# All downstream snippets use these exact objects — define once, reuse always.
# Storing as module-level variables (Kaggle notebook shares session state).

print("\n── Objects available for Snippet 3 ──")
print(f"  X_train_proc : {X_train_proc.shape}")
print(f"  X_test_proc  : {X_test_proc.shape}")
print(f"  y_train      : {y_train.shape}")
print(f"  y_test       : {y_test.shape}")
print(f"  CV           : StratifiedKFold(n_splits=5)")
print(f"  FEATURE_NAMES_OUT : {FEATURE_NAMES_OUT}")

print("\n✅ Snippet 2 complete.")
print("   Outputs: feature_importance_mi.png, feature_importance_perm.png")
print("   Proceed to Snippet 3 → XGBoost + LightGBM + LR training with 5-fold CV.")

── Columns kept for modeling ──
['gender', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'age_years', 'bmi', 'pulse_pressure']

BMI capped at 60: 35 rows adjusted

Feature groups confirmed — 5 continuous, 4 binary, 2 ordinal

Train size : 54,907  (CVD=27,163, No CVD=27,744)
Test size  : 13,727  (CVD=6,791,  No CVD=6,936)

Preprocessor fitted. Sample of scaled train data:
   age_years    bmi  ap_hi  ap_lo  pulse_pressure  gender  smoke  alco  \
0     -1.367  0.153 -0.398 -0.137          -0.458     2.0    0.0   0.0   
1     -0.179 -0.227 -0.398 -0.137          -0.458     1.0    0.0   0.0   
2     -0.832 -0.615 -0.398 -0.137          -0.458     1.0    0.0   0.0   

   active  cholesterol  gluc  
0     1.0          1.0   2.0  
1     1.0          1.0   1.0  
2     0.0          2.0   2.0  

── Mutual Information scores ──
       feature  mi_score
         ap_hi  0.118617
pulse_pressure  0.072555
         ap_lo  0.071713
     age_years  0.029752
   cholesterol 

In [5]:
# ============================================================
# SNIPPET 3 / 5 — MODEL TRAINING + STRATIFIED 5-FOLD CV
# Cardiovascular Disease Classifier | Portfolio Project
# ============================================================
# Requires: X_train_proc, X_test_proc, y_train, y_test,
#           CV, FEATURE_NAMES_OUT — all defined in Snippet 2
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings("ignore")

from sklearn.linear_model    import LogisticRegression
from sklearn.metrics         import (roc_auc_score, f1_score, accuracy_score,
                                     roc_curve, precision_recall_curve,
                                     average_precision_score)
from sklearn.model_selection import cross_validate
from xgboost                 import XGBClassifier
from lightgbm                import LGBMClassifier

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

COLORS = ["#4C72B0", "#DD8452", "#55A868"]

# ── 1. DEFINE ALL THREE MODELS ────────────────────────────────────────────────
# LightGBM: device="cpu" + n_jobs=-1 (Kaggle T4 exposes CUDA not OpenCL,
#           so LightGBM GPU backend fails — CPU with all cores is fast enough)
# XGBoost:  device="cuda" works fine (uses CUDA directly)

MODELS = {
    "Logistic Regression": LogisticRegression(
        C=0.05,           # stronger L2 vs default; handles correlated BP features
        solver="lbfgs",
        max_iter=2000,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        gamma=1,
        scale_pos_weight=1,
        use_label_encoder=False,
        eval_metric="auc",
        device="cuda",
        random_state=42,
        verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=50,
        reg_alpha=0.1,
        reg_lambda=1.0,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        device="cpu",     # OpenCL not available on Kaggle T4
        n_jobs=-1,        # use all CPU cores
        random_state=42,
        verbose=-1
    ),
}

# ── 2. STRATIFIED 5-FOLD CROSS-VALIDATION ────────────────────────────────────
print("=" * 60)
print("STRATIFIED 5-FOLD CROSS-VALIDATION")
print("=" * 60)

cv_results = {}

for name, model in MODELS.items():
    t0 = time.time()
    print(f"\nTraining {name}...")

    scores = cross_validate(
        model, X_train_proc, y_train,
        cv=CV,
        scoring=["roc_auc", "f1", "accuracy"],
        n_jobs=1,
        return_train_score=False
    )

    cv_results[name] = scores
    elapsed = time.time() - t0

    print(f"  AUC  : {scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}")
    print(f"  F1   : {scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}")
    print(f"  Acc  : {scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}")
    print(f"  Time : {elapsed:.1f}s")

# ── 3. FIT FINAL MODELS ON FULL TRAIN SET ────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL MODEL TRAINING (full train set)")
print("=" * 60)

fitted_models = {}
for name, model in MODELS.items():
    t0 = time.time()
    model.fit(X_train_proc, y_train)
    fitted_models[name] = model
    print(f"  {name} fitted in {time.time()-t0:.1f}s")

# ── 4. HELD-OUT TEST SET EVALUATION ──────────────────────────────────────────
print("\n" + "=" * 60)
print("HELD-OUT TEST SET PERFORMANCE")
print("=" * 60)

test_results = {}
for name, model in fitted_models.items():
    proba = model.predict_proba(X_test_proc)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, proba)
    f1  = f1_score(y_test, pred)
    acc = accuracy_score(y_test, pred)
    ap  = average_precision_score(y_test, proba)

    test_results[name] = {
        "AUC": auc, "F1": f1, "Accuracy": acc,
        "AP": ap, "proba": proba, "pred": pred
    }

    print(f"\n{name}")
    print(f"  AUC      : {auc:.4f}")
    print(f"  F1       : {f1:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Avg Prec : {ap:.4f}")

# ── 5. PLOT: CV SCORE COMPARISON ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("5-Fold CV performance by model", fontsize=14)

metrics     = ["test_roc_auc", "test_f1", "test_accuracy"]
metric_lbls = ["ROC-AUC", "F1 Score", "Accuracy"]

for ax, metric, label in zip(axes, metrics, metric_lbls):
    means = [cv_results[n][metric].mean() for n in MODELS]
    stds  = [cv_results[n][metric].std()  for n in MODELS]
    bars  = ax.bar(list(MODELS.keys()), means, yerr=stds,
                   color=COLORS, alpha=0.85, edgecolor="white",
                   capsize=5, error_kw={"elinewidth": 1.5})
    ax.set_title(label)
    ax.set_ylabel(label)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + s + 0.002,
                f"{m:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(min(means) - 0.05, 1.0)
    ax.tick_params(axis="x", rotation=12)

plt.tight_layout()
plt.savefig("cv_comparison.png", bbox_inches="tight")
plt.show()
print("Saved → cv_comparison.png")

# ── 6. PLOT: ROC + PRECISION-RECALL CURVES ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for (name, res), color in zip(test_results.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_test, res["proba"])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{name} (AUC={res['AUC']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — held-out test set")
ax.legend(fontsize=9)

ax = axes[1]
for (name, res), color in zip(test_results.items(), COLORS):
    prec, rec, _ = precision_recall_curve(y_test, res["proba"])
    ax.plot(rec, prec, color=color, lw=2,
            label=f"{name} (AP={res['AP']:.3f})")
ax.axhline(y_test.mean(), color="black", linestyle="--", lw=1,
           label=f"Random (AP={y_test.mean():.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — held-out test set")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("roc_pr_curves.png", bbox_inches="tight")
plt.show()
print("Saved → roc_pr_curves.png")

# ── 7. PLOT: FOLD-BY-FOLD AUC STABILITY ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
folds = list(range(1, 6))
for (name, scores), color in zip(cv_results.items(), COLORS):
    fold_aucs = scores["test_roc_auc"]
    ax.plot(folds, fold_aucs, marker="o", color=color, lw=2, label=name)
    ax.fill_between(folds,
                    fold_aucs - fold_aucs.std(),
                    fold_aucs + fold_aucs.std(),
                    alpha=0.10, color=color)
ax.set_xlabel("Fold")
ax.set_ylabel("ROC-AUC")
ax.set_title("AUC stability across CV folds")
ax.set_xticks(folds)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("cv_fold_stability.png", bbox_inches="tight")
plt.show()
print("Saved → cv_fold_stability.png")

# ── 8. SUMMARY TABLE ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL SUMMARY TABLE")
print("=" * 60)
rows = []
for name in MODELS:
    cv  = cv_results[name]
    tst = test_results[name]
    rows.append({
        "Model":         name,
        "CV AUC (mean)": f"{cv['test_roc_auc'].mean():.4f}",
        "CV AUC (±std)": f"±{cv['test_roc_auc'].std():.4f}",
        "Test AUC":      f"{tst['AUC']:.4f}",
        "Test F1":       f"{tst['F1']:.4f}",
        "Test Acc":      f"{tst['Accuracy']:.4f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

# ── 9. IDENTIFY BEST MODEL FOR SNIPPET 4 ─────────────────────────────────────
best_name  = max(test_results, key=lambda n: test_results[n]["AUC"])
best_model = fitted_models[best_name]
best_proba = test_results[best_name]["proba"]

print(f"\n🏆 Best model → {best_name}  (Test AUC = {test_results[best_name]['AUC']:.4f})")
print("   This model will be used for SHAP explainability in Snippet 4.")

print("\n✅ Snippet 3 complete.")
print("   Outputs: cv_comparison.png, roc_pr_curves.png, cv_fold_stability.png")
print("   Proceed to Snippet 4 → SHAP + calibration + confusion matrix + threshold tuning.")

STRATIFIED 5-FOLD CROSS-VALIDATION

Training Logistic Regression...
  AUC  : 0.7906 ± 0.0026
  F1   : 0.7081 ± 0.0042
  Acc  : 0.7281 ± 0.0037
  Time : 1.7s

Training XGBoost...
  AUC  : 0.7997 ± 0.0024
  F1   : 0.7203 ± 0.0022
  Acc  : 0.7351 ± 0.0014
  Time : 6.0s

Training LightGBM...
  AUC  : 0.7977 ± 0.0025
  F1   : 0.7211 ± 0.0035
  Acc  : 0.7354 ± 0.0029
  Time : 10.0s

FINAL MODEL TRAINING (full train set)
  Logistic Regression fitted in 0.2s
  XGBoost fitted in 1.2s
  LightGBM fitted in 2.0s

HELD-OUT TEST SET PERFORMANCE

Logistic Regression
  AUC      : 0.7933
  F1       : 0.7052
  Accuracy : 0.7248
  Avg Prec : 0.7769

XGBoost
  AUC      : 0.8003
  F1       : 0.7189
  Accuracy : 0.7327
  Avg Prec : 0.7816

LightGBM
  AUC      : 0.7985
  F1       : 0.7186
  Accuracy : 0.7321
  Avg Prec : 0.7800
Saved → cv_comparison.png
Saved → roc_pr_curves.png
Saved → cv_fold_stability.png

FINAL SUMMARY TABLE
              Model CV AUC (mean) CV AUC (±std) Test AUC Test F1 Test Acc
Logist

In [6]:
# ============================================================
# SNIPPET 3B / 5 — OPTUNA TUNING + SOFT VOTING ENSEMBLE
# Cardiovascular Disease Classifier | Portfolio Project
# ============================================================
# Requires: X_train_proc, X_test_proc, y_train, y_test,
#           CV, fitted_models, test_results, cv_results
#           — all defined in Snippet 3
# ============================================================

import optuna
import numpy as np
import pandas as pd
import warnings, time
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

from xgboost              import XGBClassifier
from lightgbm             import LGBMClassifier
from sklearn.metrics      import roc_auc_score, f1_score, accuracy_score, average_precision_score
from sklearn.model_selection import cross_val_score

# ── 1. OPTUNA OBJECTIVE — XGBOOST ────────────────────────────────────────────
# We tune XGBoost (best single model at 0.8003) with 60 trials.
# Search space is bounded by what makes clinical/statistical sense:
#   n_estimators: 300–1000 — below 300 underfits; above 1000 = diminishing returns
#   learning_rate: 0.01–0.15 log-scale — log because effect is multiplicative
#   max_depth: 3–7 — tabular medical data rarely benefits past 6
#   subsample/colsample: 0.6–1.0 — standard stochastic boosting range
#   min_child_weight: 5–50 — with 54k rows, 50 still gives ~1000+ leaves possible
#   gamma: 0–5 — pruning; 0=no pruning, 5=very conservative
#   reg_alpha: 0–2 (L1), reg_lambda: 0.5–5 (L2) — regularization search

def xgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 7),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 5, 50),
        "gamma":             trial.suggest_float("gamma", 0, 5),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0, 2),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.5, 5),
        "scale_pos_weight":  1,
        "use_label_encoder": False,
        "eval_metric":       "auc",
        "device":            "cuda",
        "random_state":      42,
        "verbosity":         0,
    }
    model = XGBClassifier(**params)
    # Use 3-fold inside Optuna (faster) — full 5-fold on the winner only
    scores = cross_val_score(model, X_train_proc, y_train,
                             cv=3, scoring="roc_auc", n_jobs=1)
    return scores.mean()

print("Running Optuna XGBoost search — 60 trials (3-fold each)...")
t0 = time.time()
xgb_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f"Done in {time.time()-t0:.0f}s")
print(f"Best 3-fold AUC: {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}")

# ── 2. OPTUNA OBJECTIVE — LIGHTGBM ───────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 80),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "feature_fraction":  trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq":      trial.suggest_int("bagging_freq", 1, 10),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0, 2),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.5, 5),
        "device":            "cpu",
        "n_jobs":            -1,
        "random_state":      42,
        "verbose":           -1,
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(model, X_train_proc, y_train,
                             cv=3, scoring="roc_auc", n_jobs=1)
    return scores.mean()

print("\nRunning Optuna LightGBM search — 60 trials (3-fold each)...")
t0 = time.time()
lgbm_study = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=42))
lgbm_study.optimize(lgbm_objective, n_trials=60, show_progress_bar=True)
print(f"Done in {time.time()-t0:.0f}s")
print(f"Best 3-fold AUC: {lgbm_study.best_value:.4f}")
print(f"Best params: {lgbm_study.best_params}")

# ── 3. RETRAIN TUNED MODELS ON FULL TRAIN SET ─────────────────────────────────
print("\nRetraining tuned models on full train set...")

xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=1,
    use_label_encoder=False,
    eval_metric="auc",
    device="cuda",
    random_state=42,
    verbosity=0
)
xgb_tuned.fit(X_train_proc, y_train)

lgbm_tuned = LGBMClassifier(
    **lgbm_study.best_params,
    device="cpu",
    n_jobs=-1,
    random_state=42,
    verbose=-1
)
lgbm_tuned.fit(X_train_proc, y_train)

# ── 4. VALIDATE TUNED MODELS WITH FULL 5-FOLD CV ─────────────────────────────
print("\nFull 5-fold CV on tuned models...")
for name, model in [("XGBoost (tuned)", xgb_tuned), ("LightGBM (tuned)", lgbm_tuned)]:
    scores = cross_val_score(model, X_train_proc, y_train,
                             cv=CV, scoring="roc_auc", n_jobs=1)
    print(f"  {name}: {scores.mean():.4f} ± {scores.std():.4f}")

# ── 5. SOFT VOTING ENSEMBLE ───────────────────────────────────────────────────
# Average predicted probabilities from all 5 models.
# Soft voting outperforms any single model when models make different errors.
# LR captures linear boundaries; XGBoost/LightGBM capture non-linear ones.
# Averaging probabilities is more stable than majority voting on hard labels.

print("\nBuilding soft voting ensemble (all 5 models)...")

proba_lr   = fitted_models["Logistic Regression"].predict_proba(X_test_proc)[:, 1]
proba_xgb  = fitted_models["XGBoost"].predict_proba(X_test_proc)[:, 1]
proba_lgbm = fitted_models["LightGBM"].predict_proba(X_test_proc)[:, 1]
proba_xgbt = xgb_tuned.predict_proba(X_test_proc)[:, 1]
proba_lgbt = lgbm_tuned.predict_proba(X_test_proc)[:, 1]

# Equal weights — justified because all models are within 0.002 AUC of each other
# Weighted ensemble only helps when one model is clearly dominant (>0.01 gap)
ensemble_proba = (proba_lr + proba_xgb + proba_lgbm + proba_xgbt + proba_lgbt) / 5
ensemble_pred  = (ensemble_proba >= 0.5).astype(int)

# ── 6. FINAL COMPARISON TABLE ─────────────────────────────────────────────────
print("\n" + "=" * 65)
print("FINAL RESULTS — ALL MODELS + ENSEMBLE")
print("=" * 65)

all_results = {
    "LR (baseline)":      proba_lr,
    "XGBoost (default)":  proba_xgb,
    "LightGBM (default)": proba_lgbm,
    "XGBoost (tuned)":    proba_xgbt,
    "LightGBM (tuned)":   proba_lgbt,
    "Soft Ensemble":      ensemble_proba,
}

rows = []
for name, proba in all_results.items():
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "Model":     name,
        "Test AUC":  f"{roc_auc_score(y_test, proba):.4f}",
        "Test F1":   f"{f1_score(y_test, pred):.4f}",
        "Test Acc":  f"{accuracy_score(y_test, pred):.4f}",
        "Avg Prec":  f"{average_precision_score(y_test, proba):.4f}",
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

# ── 7. UPDATE BEST MODEL FOR SNIPPET 4 ────────────────────────────────────────
# Pick whichever has the highest test AUC for SHAP
auc_scores = {name: roc_auc_score(y_test, proba)
              for name, proba in all_results.items()}
best_name_final = max(auc_scores, key=auc_scores.get)

# For SHAP we need a tree model — if ensemble wins, use best tree underneath it
shap_model = xgb_tuned   # XGBoost tuned: SHAP native support + CUDA
shap_proba = proba_xgbt

print(f"\n🏆 Best overall  → {best_name_final} (AUC={auc_scores[best_name_final]:.4f})")
print(f"🔍 SHAP model    → XGBoost (tuned) — native TreeExplainer support")
print(f"   Report AUC in README as ensemble = {auc_scores['Soft Ensemble']:.4f}")

# Store ensemble proba for Snippet 4 threshold tuning
best_ensemble_proba = ensemble_proba

print("\n✅ Snippet 3B complete.")
print("   Proceed to Snippet 4 → SHAP + calibration + confusion matrix + threshold tuning.")

Running Optuna XGBoost search — 60 trials (3-fold each)...


  0%|          | 0/60 [00:00<?, ?it/s]

Done in 189s
Best 3-fold AUC: 0.8016
Best params: {'n_estimators': 508, 'learning_rate': 0.014772856542828144, 'max_depth': 5, 'subsample': 0.7621249295430792, 'colsample_bytree': 0.8426740033019516, 'min_child_weight': 28, 'gamma': 1.5749683437666597, 'reg_alpha': 1.4598873465725903, 'reg_lambda': 1.8403004768385594}

Running Optuna LightGBM search — 60 trials (3-fold each)...


  0%|          | 0/60 [00:00<?, ?it/s]

Done in 367s
Best 3-fold AUC: 0.8014
Best params: {'n_estimators': 318, 'learning_rate': 0.020680651570084026, 'num_leaves': 22, 'min_child_samples': 52, 'feature_fraction': 0.6858804952725545, 'bagging_fraction': 0.6476575183812997, 'bagging_freq': 2, 'reg_alpha': 0.327745171019019, 'reg_lambda': 3.485204482638237}

Retraining tuned models on full train set...

Full 5-fold CV on tuned models...
  XGBoost (tuned): 0.8012 ± 0.0026
  LightGBM (tuned): 0.8013 ± 0.0026

Building soft voting ensemble (all 5 models)...

FINAL RESULTS — ALL MODELS + ENSEMBLE
             Model Test AUC Test F1 Test Acc Avg Prec
     LR (baseline)   0.7933  0.7052   0.7248   0.7769
 XGBoost (default)   0.8003  0.7189   0.7327   0.7816
LightGBM (default)   0.7985  0.7186   0.7321   0.7800
   XGBoost (tuned)   0.8017  0.7193   0.7333   0.7834
  LightGBM (tuned)   0.8021  0.7211   0.7341   0.7843
     Soft Ensemble   0.8020  0.7184   0.7326   0.7844

🏆 Best overall  → LightGBM (tuned) (AUC=0.8021)
🔍 SHAP model   

In [7]:
# ============================================================
# SNIPPET 3C / 5 — DEEP CLINICAL FEATURE ENGINEERING
# Cardiovascular Disease Classifier | Portfolio Project
# ============================================================
# Requires: df (cleaned, from Snippet 1)
# Rebuilds everything from df → replaces Snippets 2 & 3 objects
# Run this as ONE cell, it is fully self-contained.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, time
warnings.filterwarnings("ignore")

from sklearn.model_selection  import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing    import StandardScaler
from sklearn.compose          import ColumnTransformer
from sklearn.metrics          import (roc_auc_score, f1_score, accuracy_score,
                                      average_precision_score)
from xgboost                  import XGBClassifier
from lightgbm                 import LGBMClassifier

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# ── 1. DEEP CLINICAL FEATURE ENGINEERING ─────────────────────────────────────
# Every feature below is grounded in published clinical guidelines.
# These are the exact variables a cardiologist computes from raw vitals.
# This is what separates a portfolio project from a tutorial.

df_fe = df.copy()

# ── 1a. JNC-8 HYPERTENSION STAGING ───────────────────────────────────────────
# Joint National Committee 8th report — gold standard BP classification.
# Encodes the NON-LINEAR relationship between BP and CVD risk.
# A jump from stage 1→2 hypertension has a disproportionate effect on risk
# that raw ap_hi values (linear) cannot represent as efficiently.
#   0 = Normal          : systolic <120  AND diastolic <80
#   1 = Elevated        : systolic 120–129 AND diastolic <80
#   2 = Hypertension S1 : systolic 130–139 OR diastolic 80–89
#   3 = Hypertension S2 : systolic 140–179 OR diastolic 90–119
#   4 = Hypertensive crisis: systolic ≥180 OR diastolic ≥120

def jnc8_stage(row):
    s, d = row["ap_hi"], row["ap_lo"]
    if s >= 180 or d >= 120:  return 4
    if s >= 140 or d >= 90:   return 3
    if s >= 130 or d >= 80:   return 2
    if s >= 120 and d < 80:   return 1
    return 0

df_fe["bp_stage"] = df_fe.apply(jnc8_stage, axis=1)
print("BP stage distribution:")
print(df_fe["bp_stage"].value_counts().sort_index())

# ── 1b. WHO BMI CLASSIFICATION ────────────────────────────────────────────────
# WHO 2000 cut-points — standard in cardiovascular risk literature.
# Again encodes non-linearity: obese class II+ has exponentially higher risk.
#   0 = Underweight  : BMI < 18.5
#   1 = Normal       : 18.5 – 24.9
#   2 = Overweight   : 25.0 – 29.9
#   3 = Obese I      : 30.0 – 34.9
#   4 = Obese II+    : ≥ 35.0

def who_bmi(bmi):
    if bmi < 18.5: return 0
    if bmi < 25.0: return 1
    if bmi < 30.0: return 2
    if bmi < 35.0: return 3
    return 4

df_fe["bmi_category"] = df_fe["bmi"].apply(who_bmi)
print("\nBMI category distribution:")
print(df_fe["bmi_category"].value_counts().sort_index())

# ── 1c. MEAN ARTERIAL PRESSURE (MAP) ─────────────────────────────────────────
# MAP = diastolic + (pulse_pressure / 3)
# Standard formula used in ICU and cardiology for perfusion pressure.
# Captures the average pressure the heart works against — more informative
# than systolic or diastolic alone for sustained cardiovascular load.
df_fe["map"] = df_fe["ap_lo"] + (df_fe["pulse_pressure"] / 3)

# ── 1d. AGE × SYSTOLIC BP INTERACTION ────────────────────────────────────────
# The combination of old age + high BP is superadditive in CVD risk —
# arterial stiffness compounds with sustained pressure over decades.
# Published Framingham risk score includes this interaction explicitly.
# Standardize both before multiplying to prevent scale dominance.
age_z  = (df_fe["age_years"] - df_fe["age_years"].mean()) / df_fe["age_years"].std()
aphi_z = (df_fe["ap_hi"]    - df_fe["ap_hi"].mean())     / df_fe["ap_hi"].std()
df_fe["age_bp_interaction"] = age_z * aphi_z

# ── 1e. COMBINED METABOLIC RISK FLAG ──────────────────────────────────────────
# High cholesterol AND high glucose simultaneously = metabolic syndrome pattern.
# Both above normal (≥2) together is a stronger signal than either alone.
# Binary flag: 1 = both elevated, 0 = otherwise.
df_fe["metabolic_risk"] = (
    (df_fe["cholesterol"] >= 2) & (df_fe["gluc"] >= 2)
).astype(int)

# ── 1f. LIFESTYLE BURDEN SCORE ────────────────────────────────────────────────
# Additive count of negative lifestyle factors: smoking + alcohol + inactive.
# While individual lifestyle features scored near-zero MI in Snippet 2,
# their COMBINATION may carry signal (a person who smokes, drinks, AND is
# inactive is qualitatively different from someone with only one factor).
# Range 0–3.
df_fe["lifestyle_burden"] = (
    df_fe["smoke"] + df_fe["alco"] + (1 - df_fe["active"])
)

# ── 1g. HYPERTENSIVE + OBESE FLAG ─────────────────────────────────────────────
# Hypertension stage ≥2 AND BMI category ≥3 = highest-risk clinical cluster.
# This interaction is explicitly listed in ACC/AHA CVD risk guidelines.
df_fe["htn_obese"] = (
    (df_fe["bp_stage"] >= 2) & (df_fe["bmi_category"] >= 3)
).astype(int)

# ── 1h. PULSE PRESSURE CATEGORY ───────────────────────────────────────────────
# Pulse pressure >60 mmHg = "wide pulse pressure" — marker of arterial
# stiffness and independent predictor of CVD in adults >50 (ESC guidelines).
#   0 = Normal      : PP ≤ 40
#   1 = Borderline  : PP 41–60
#   2 = Wide        : PP > 60
def pp_category(pp):
    if pp <= 40: return 0
    if pp <= 60: return 1
    return 2

df_fe["pp_category"] = df_fe["pulse_pressure"].apply(pp_category)

# ── 1i. AGE GROUP ─────────────────────────────────────────────────────────────
# ESC risk guidelines stratify by decade: <40, 40–49, 50–59, 60+
# Encodes the non-linear age-risk relationship.
def age_group(age):
    if age < 40: return 0
    if age < 50: return 1
    if age < 60: return 2
    return 3

df_fe["age_group"] = df_fe["age_years"].apply(age_group)

print(f"\nTotal features after engineering: {df_fe.shape[1]}")
print("New features added:")
new_feats = ["bp_stage","bmi_category","map","age_bp_interaction",
             "metabolic_risk","lifestyle_burden","htn_obese","pp_category","age_group"]
for f in new_feats:
    print(f"  {f}: min={df_fe[f].min():.2f}, max={df_fe[f].max():.2f}, "
          f"mean={df_fe[f].mean():.3f}")

# ── 2. DEFINE FINAL FEATURE SET ───────────────────────────────────────────────
# Keep original features + all engineered ones.
# Drop raw weight/height (replaced by bmi), raw age days (replaced by age_years).
# Keep ap_hi, ap_lo, pulse_pressure alongside bp_stage — tree models benefit
# from having both the raw values AND the categorized version simultaneously.

CONTINUOUS_FE = ["age_years", "bmi", "ap_hi", "ap_lo", "pulse_pressure",
                 "map", "age_bp_interaction"]
ORDINAL_FE    = ["cholesterol", "gluc", "bp_stage", "bmi_category",
                 "pp_category", "age_group"]
BINARY_FE     = ["gender", "smoke", "alco", "active",
                 "metabolic_risk", "lifestyle_burden", "htn_obese"]
TARGET        = "cardio"

FEATURE_NAMES_FE = CONTINUOUS_FE + ORDINAL_FE + BINARY_FE
print(f"\nFinal feature count: {len(FEATURE_NAMES_FE)}")

X_fe = df_fe[FEATURE_NAMES_FE]
y_fe = df_fe[TARGET]

# ── 3. TRAIN/TEST SPLIT ───────────────────────────────────────────────────────
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.20, stratify=y_fe, random_state=42
)

# ── 4. PREPROCESSING PIPELINE ────────────────────────────────────────────────
preprocessor_fe = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), CONTINUOUS_FE),
        ("pass",  "passthrough",   ORDINAL_FE + BINARY_FE),
    ],
    remainder="drop"
)

X_train_fe_proc = preprocessor_fe.fit_transform(X_train_fe)
X_test_fe_proc  = preprocessor_fe.transform(X_test_fe)

# ── 5. REUSE BEST TUNED PARAMS FROM OPTUNA (Snippet 3B) ──────────────────────
# We already have the best hyperparameters — no need to re-tune.
# The richer feature set is the intervention; params stay fixed for fair comparison.

from sklearn.model_selection import StratifiedKFold
CV_FE = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_fe = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=1,
    use_label_encoder=False,
    eval_metric="auc",
    device="cuda",
    random_state=42,
    verbosity=0
)

lgbm_fe = LGBMClassifier(
    **lgbm_study.best_params,
    device="cpu",
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

# ── 6. 5-FOLD CV ON ENGINEERED FEATURES ──────────────────────────────────────
print("\n" + "=" * 60)
print("5-FOLD CV — ENGINEERED FEATURES")
print("=" * 60)

fe_cv_results = {}
fe_fitted     = {}

for name, model in [("XGBoost (FE)", xgb_fe), ("LightGBM (FE)", lgbm_fe)]:
    t0 = time.time()
    scores = cross_validate(model, X_train_fe_proc, y_train_fe,
                            cv=CV_FE,
                            scoring=["roc_auc", "f1", "accuracy"],
                            n_jobs=1)
    fe_cv_results[name] = scores
    print(f"\n{name}")
    print(f"  AUC  : {scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}")
    print(f"  F1   : {scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}")
    print(f"  Acc  : {scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}")
    print(f"  Time : {time.time()-t0:.1f}s")

    # Fit on full train for test eval
    model.fit(X_train_fe_proc, y_train_fe)
    fe_fitted[name] = model

# ── 7. SOFT ENSEMBLE — ENGINEERED FEATURES ───────────────────────────────────
proba_xgb_fe  = fe_fitted["XGBoost (FE)"].predict_proba(X_test_fe_proc)[:, 1]
proba_lgbm_fe = fe_fitted["LightGBM (FE)"].predict_proba(X_test_fe_proc)[:, 1]
ensemble_fe   = (proba_xgb_fe + proba_lgbm_fe) / 2

# ── 8. FULL COMPARISON TABLE ──────────────────────────────────────────────────
print("\n" + "=" * 68)
print("BEFORE vs AFTER FEATURE ENGINEERING — TEST SET")
print("=" * 68)

comparison = {
    # Before FE (from Snippet 3B)
    "XGBoost (tuned, no FE)":   proba_xgbt,
    "LightGBM (tuned, no FE)":  proba_lgbt,
    "Ensemble (no FE)":         best_ensemble_proba,
    # After FE
    "XGBoost (tuned + FE)":     proba_xgb_fe,
    "LightGBM (tuned + FE)":    proba_lgbm_fe,
    "Ensemble (FE)":            ensemble_fe,
}

rows = []
for name, proba in comparison.items():
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "Model":    name,
        "AUC":      f"{roc_auc_score(y_test_fe, proba):.4f}",
        "F1":       f"{f1_score(y_test_fe, pred):.4f}",
        "Accuracy": f"{accuracy_score(y_test_fe, pred):.4f}",
        "Avg Prec": f"{average_precision_score(y_test_fe, proba):.4f}",
    })

print(pd.DataFrame(rows).to_string(index=False))

# ── 9. PLOT: BEFORE vs AFTER AUC ─────────────────────────────────────────────
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(9, 6))
styles = {
    "XGBoost (tuned, no FE)":  ("--", "#DD8452", 1.5),
    "LightGBM (tuned, no FE)": ("--", "#55A868", 1.5),
    "Ensemble (no FE)":        ("--", "#4C72B0", 2.0),
    "XGBoost (tuned + FE)":    ("-",  "#DD8452", 1.5),
    "LightGBM (tuned + FE)":   ("-",  "#55A868", 1.5),
    "Ensemble (FE)":           ("-",  "#4C72B0", 2.5),
}
for name, proba in comparison.items():
    ls, color, lw = styles[name]
    fpr, tpr, _   = roc_curve(y_test_fe, proba)
    auc           = roc_auc_score(y_test_fe, proba)
    ax.plot(fpr, tpr, linestyle=ls, color=color, lw=lw,
            label=f"{name} (AUC={auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — before vs after feature engineering")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("roc_before_after_fe.png", bbox_inches="tight")
plt.show()
print("Saved → roc_before_after_fe.png")

# ── 10. IDENTIFY BEST MODEL FOR SNIPPET 4 ────────────────────────────────────
all_aucs = {n: roc_auc_score(y_test_fe, p) for n, p in comparison.items()}
best_fe_name = max(all_aucs, key=all_aucs.get)

# For SHAP: always use a single tree model (TreeExplainer needs it)
# Use whichever FE tree model scored higher
xgb_auc  = roc_auc_score(y_test_fe, proba_xgb_fe)
lgbm_auc = roc_auc_score(y_test_fe, proba_lgbm_fe)
shap_model_fe    = fe_fitted["XGBoost (FE)"] if xgb_auc >= lgbm_auc else fe_fitted["LightGBM (FE)"]
shap_model_name  = "XGBoost (FE)" if xgb_auc >= lgbm_auc else "LightGBM (FE)"
shap_proba_fe    = proba_xgb_fe   if xgb_auc >= lgbm_auc else proba_lgbm_fe

# Store for Snippet 4
best_ensemble_fe_proba = ensemble_fe
FEATURE_NAMES_OUT      = FEATURE_NAMES_FE   # update global for SHAP

print(f"\n🏆 Best model     → {best_fe_name}  (AUC={all_aucs[best_fe_name]:.4f})")
print(f"🔍 SHAP model     → {shap_model_name}")
print(f"📊 README AUC     → Ensemble (FE) = {all_aucs['Ensemble (FE)']:.4f}")

print("\n✅ Snippet 3C complete.")
print("   Output: roc_before_after_fe.png")
print("   Proceed to Snippet 4 → SHAP + calibration + confusion matrix + threshold tuning.")

BP stage distribution:
bp_stage
0     9551
1     3108
2    32444
3    22556
4      975
Name: count, dtype: int64

BMI category distribution:
bmi_category
0      630
1    25294
2    24707
3    11967
4     6036
Name: count, dtype: int64

Total features after engineering: 25
New features added:
  bp_stage: min=0.00, max=4.00, mean=2.033
  bmi_category: min=0.00, max=4.00, mean=1.963
  map: min=46.67, max=173.33, mean=96.424
  age_bp_interaction: min=-9.04, max=9.11, mean=0.210
  metabolic_risk: min=0.00, max=1.00, mean=0.096
  lifestyle_burden: min=0.00, max=3.00, mean=0.338
  htn_obese: min=0.00, max=1.00, mean=0.234
  pp_category: min=0.00, max=2.00, mean=0.461
  age_group: min=0.00, max=3.00, mean=1.857

Final feature count: 20

5-FOLD CV — ENGINEERED FEATURES

XGBoost (FE)
  AUC  : 0.8010 ± 0.0026
  F1   : 0.7200 ± 0.0032
  Acc  : 0.7355 ± 0.0027
  Time : 7.5s

LightGBM (FE)
  AUC  : 0.8011 ± 0.0026
  F1   : 0.7212 ± 0.0031
  Acc  : 0.7354 ± 0.0025
  Time : 8.2s

BEFORE vs AFTER FEATU

In [8]:
# ============================================================
# SNIPPET 4 / 5 — EXPLAINABILITY + CALIBRATION + EVALUATION
# Cardiovascular Disease Classifier | Portfolio Project
# ============================================================
# Requires ALL objects from Snippets 1–3C in session memory.
# Uses: shap_model_fe, X_test_fe_proc, X_test_fe, y_test_fe,
#       FEATURE_NAMES_FE, best_ensemble_fe_proba, fe_fitted,
#       proba_xgb_fe, proba_lgbm_fe
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (confusion_matrix, classification_report,
                                      roc_auc_score, roc_curve,
                                      brier_score_loss,
                                      precision_recall_curve, f1_score,
                                      accuracy_score, recall_score,
                                      precision_score)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

plt.rcParams.update({
    "figure.dpi"         : 120,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "font.family"        : "DejaVu Sans",
    "axes.titlesize"     : 13,
    "axes.labelsize"     : 11,
})
COLORS = ["#4C72B0", "#DD8452", "#55A868"]

# ── 1. SHAP — TREE EXPLAINER ──────────────────────────────────────────────────
# TreeExplainer computes exact Shapley values for tree-based models in
# polynomial time. It is NOT an approximation — it's the theoretically
# correct attribution of each feature's marginal contribution.
# We use the FE XGBoost model for SHAP so the richer feature set
# (bp_stage, map, age_bp_interaction etc.) appears in the plots —
# giving a clinically richer story even if raw AUC is identical.

print("Computing SHAP values (TreeExplainer)...")
explainer   = shap.TreeExplainer(shap_model_fe)
shap_values = explainer.shap_values(X_test_fe_proc)
print(f"SHAP matrix shape: {shap_values.shape}")  # (n_test, n_features)

# Feature names aligned to preprocessor output order
# CONTINUOUS_FE scaled → ORDINAL_FE + BINARY_FE passthrough
feat_names = FEATURE_NAMES_FE   # defined at end of Snippet 3C

# ── 2. PLOT: SHAP SUMMARY (BEE SWARM) ────────────────────────────────────────
# Each dot = one patient. X-axis = SHAP value (impact on log-odds of CVD).
# Colour = feature value (red=high, blue=low).
# This is the single most powerful explainability plot — shows both
# direction AND magnitude of every feature for every patient.

print("\nPlotting SHAP summary (beeswarm)...")
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test_fe_proc,
    feature_names=feat_names,
    show=False, plot_size=None,
    color_bar_label="Feature value"
)
plt.title("SHAP feature impact — cardiovascular disease prediction", pad=12)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", bbox_inches="tight")
plt.show()
print("Saved → shap_beeswarm.png")

# ── 3. PLOT: SHAP BAR — MEAN ABSOLUTE IMPACT ─────────────────────────────────
# Mean |SHAP| per feature = average magnitude of contribution across all patients.
# This is the portfolio-ready "feature importance" chart —
# more honest than XGBoost's built-in gain importance which is biased
# toward high-cardinality features.

mean_shap = np.abs(shap_values).mean(axis=0)
shap_bar_df = pd.DataFrame({
    "feature":    feat_names,
    "mean_shap":  mean_shap
}).sort_values("mean_shap", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = ["#378ADD" if v > shap_bar_df["mean_shap"].median()
              else "#AAAAAA" for v in shap_bar_df["mean_shap"]]
ax.barh(shap_bar_df["feature"], shap_bar_df["mean_shap"],
        color=colors_bar, edgecolor="white")
ax.axvline(shap_bar_df["mean_shap"].median(), color="red",
           linestyle="--", lw=1, label="Median importance")
ax.set_xlabel("Mean |SHAP value| (average impact on model output)")
ax.set_title("Feature importance — SHAP mean absolute values")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("shap_bar.png", bbox_inches="tight")
plt.show()
print("Saved → shap_bar.png")

# ── 4. PLOT: SHAP WATERFALL — SINGLE PATIENT EXPLANATION ─────────────────────
# Pick one high-risk patient (predicted probability > 0.80) and one
# low-risk patient (predicted probability < 0.20) for side-by-side waterfall.
# This is what you show in a demo: "here's WHY the model flagged this patient."

proba_test = shap_model_fe.predict_proba(X_test_fe_proc)[:, 1]

high_risk_idx = np.where(proba_test > 0.80)[0][0]
low_risk_idx  = np.where(proba_test < 0.20)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("SHAP waterfall — individual patient explanations", fontsize=13)

for ax, idx, label, color in zip(
    axes,
    [high_risk_idx, low_risk_idx],
    [f"High-risk patient (P={proba_test[high_risk_idx]:.2f})",
     f"Low-risk patient  (P={proba_test[low_risk_idx]:.2f})"],
    ["#DD8452", "#4C72B0"]
):
    patient_shap = shap_values[idx]
    patient_vals = X_test_fe_proc[idx]

    # Sort by absolute SHAP value, show top 10
    order   = np.argsort(np.abs(patient_shap))[-10:]
    feats   = [feat_names[i] for i in order]
    shapv   = patient_shap[order]
    featv   = patient_vals[order]

    bar_colors = ["#DD8452" if v > 0 else "#4C72B0" for v in shapv]
    bars = ax.barh(feats, shapv, color=bar_colors, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("SHAP value (contribution to CVD prediction)")
    ax.set_title(label, color=color, fontweight="bold")

    # Annotate with raw feature value
    for bar, fv in zip(bars, featv):
        ax.text(bar.get_width() + 0.002 if bar.get_width() >= 0
                else bar.get_width() - 0.002,
                bar.get_y() + bar.get_height() / 2,
                f"{fv:.1f}",
                va="center",
                ha="left" if bar.get_width() >= 0 else "right",
                fontsize=8, color="black")

plt.tight_layout()
plt.savefig("shap_waterfall.png", bbox_inches="tight")
plt.show()
print("Saved → shap_waterfall.png")

# ── 5. PLOT: SHAP DEPENDENCE — ap_hi vs age_bp_interaction ───────────────────
# Dependence plot: X = feature value, Y = SHAP value, colour = interaction feature.
# Shows HOW the model uses systolic BP — at what threshold SHAP value jumps,
# and whether the effect is moderated by age (colour).
# This is the plot that proves your model learned clinical knowledge.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, feat, interact in zip(
    axes,
    ["ap_hi", "age_years"],
    ["age_years", "ap_hi"]
):
    fidx  = feat_names.index(feat)
    iidx  = feat_names.index(interact)
    xvals = X_test_fe_proc[:, fidx]
    yvals = shap_values[:, fidx]
    cvals = X_test_fe_proc[:, iidx]

    sc = ax.scatter(xvals, yvals, c=cvals, cmap="coolwarm",
                    alpha=0.3, s=8, rasterized=True)
    plt.colorbar(sc, ax=ax, label=interact)
    ax.axhline(0, color="black", lw=0.8, linestyle="--")
    ax.set_xlabel(feat)
    ax.set_ylabel(f"SHAP value for {feat}")
    ax.set_title(f"SHAP dependence: {feat} (coloured by {interact})")

plt.tight_layout()
plt.savefig("shap_dependence.png", bbox_inches="tight")
plt.show()
print("Saved → shap_dependence.png")

# ── 6. CALIBRATION CURVE ──────────────────────────────────────────────────────
# A calibrated model: when it says P=0.7, roughly 70% of those patients
# actually have CVD. This matters enormously in clinical deployment.
# Brier score = mean squared error of probability predictions (lower = better).
# Perfect calibration = 0.0, random = 0.25 (for balanced classes).
#
# We also fit an isotonic regression calibrator on the ensemble probabilities
# and show the improvement — this is a technique rarely shown in student projects.

print("\nCalibrating ensemble probabilities (isotonic regression)...")
from sklearn.linear_model import LogisticRegression as LR

# Fit isotonic calibrator on a held-out portion of train
# Use 20% of train as calibration set to avoid leakage
from sklearn.model_selection import train_test_split as tts
X_cal, X_val_cal, y_cal, y_val_cal = tts(
    X_train_fe_proc, y_train_fe,
    test_size=0.20, stratify=y_train_fe, random_state=0
)

# Get ensemble proba on calibration validation set
p_xgb_cal = fe_fitted["XGBoost (FE)"].predict_proba(X_val_cal)[:, 1]
p_lgb_cal  = fe_fitted["LightGBM (FE)"].predict_proba(X_val_cal)[:, 1]
ens_cal    = (p_xgb_cal + p_lgb_cal) / 2

# Fit isotonic calibrator
from sklearn.isotonic import IsotonicRegression
iso_cal = IsotonicRegression(out_of_bounds="clip")
iso_cal.fit(ens_cal, y_val_cal)

# Apply to test set
calibrated_proba = iso_cal.predict(best_ensemble_fe_proba)

# Calibration curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for proba, label, color, ls in [
    (proba_xgb_fe,          "XGBoost (FE)",       "#DD8452", "--"),
    (proba_lgbm_fe,         "LightGBM (FE)",      "#55A868", "--"),
    (best_ensemble_fe_proba,"Ensemble (raw)",      "#4C72B0", "-"),
    (calibrated_proba,      "Ensemble (isotonic)","#9B59B6", "-"),
]:
    frac_pos, mean_pred = calibration_curve(y_test_fe, proba, n_bins=12)
    bs = brier_score_loss(y_test_fe, proba)
    ax.plot(mean_pred, frac_pos, marker="o", markersize=4,
            color=color, linestyle=ls, lw=1.8,
            label=f"{label} (Brier={bs:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual CVD rate)")
ax.set_title("Calibration curves")
ax.legend(fontsize=8)

# Probability distribution
ax = axes[1]
ax.hist(best_ensemble_fe_proba[y_test_fe == 0], bins=40,
        alpha=0.55, color="#4C72B0", label="No CVD", density=True)
ax.hist(best_ensemble_fe_proba[y_test_fe == 1], bins=40,
        alpha=0.55, color="#DD8452", label="CVD",    density=True)
ax.set_xlabel("Predicted probability of CVD")
ax.set_ylabel("Density")
ax.set_title("Predicted probability distribution by true class")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("calibration.png", bbox_inches="tight")
plt.show()
print("Saved → calibration.png")

# ── 7. THRESHOLD OPTIMISATION ────────────────────────────────────────────────
# Default threshold = 0.5 assumes equal cost of FP and FN.
# In cardiology, a false negative (missing a CVD patient) is far more costly
# than a false positive (unnecessary follow-up).
# We find the threshold that maximises F1, and separately the threshold
# that achieves recall ≥ 0.80 (clinical sensitivity target).

thresholds  = np.arange(0.20, 0.80, 0.01)
f1_scores   = []
recall_vals = []
prec_vals   = []
acc_vals    = []

from sklearn.metrics import recall_score, precision_score

for t in thresholds:
    pred_t = (calibrated_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test_fe, pred_t))
    recall_vals.append(recall_score(y_test_fe, pred_t))
    prec_vals.append(precision_score(y_test_fe, pred_t))
    acc_vals.append(accuracy_score(y_test_fe, pred_t))

best_f1_thresh    = thresholds[np.argmax(f1_scores)]
# Clinical threshold: first threshold where recall >= 0.80
clinical_thresh   = next((t for t, r in zip(thresholds, recall_vals)
                          if r >= 0.80), thresholds[0])

print(f"\nThreshold analysis:")
print(f"  Default  (t=0.50) → F1={f1_score(y_test_fe,(calibrated_proba>=0.50).astype(int)):.4f}, "
      f"Recall={recall_score(y_test_fe,(calibrated_proba>=0.50).astype(int)):.4f}")
print(f"  Best F1  (t={best_f1_thresh:.2f}) → F1={max(f1_scores):.4f}, "
      f"Recall={recall_vals[np.argmax(f1_scores)]:.4f}")
print(f"  Clinical (t={clinical_thresh:.2f}) → F1={f1_scores[list(thresholds).index(clinical_thresh)]:.4f}, "
      f"Recall=≥0.80")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_scores,   color="#4C72B0", lw=2,   label="F1 Score")
ax.plot(thresholds, recall_vals, color="#DD8452", lw=2,   label="Recall (Sensitivity)")
ax.plot(thresholds, prec_vals,   color="#55A868", lw=2,   label="Precision")
ax.plot(thresholds, acc_vals,    color="#9B59B6", lw=1.5, label="Accuracy", linestyle="--")
ax.axvline(best_f1_thresh,  color="#4C72B0", linestyle=":", lw=1.5,
           label=f"Best F1 threshold ({best_f1_thresh:.2f})")
ax.axvline(clinical_thresh, color="#DD8452", linestyle=":", lw=1.5,
           label=f"Clinical threshold ({clinical_thresh:.2f}, recall≥0.80)")
ax.axvline(0.50,            color="black",   linestyle="--", lw=1,
           label="Default threshold (0.50)")
ax.set_xlabel("Classification threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold optimisation — calibrated ensemble")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig("threshold_analysis.png", bbox_inches="tight")
plt.show()
print("Saved → threshold_analysis.png")

# ── 8. CONFUSION MATRICES — DEFAULT vs CLINICAL THRESHOLD ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Confusion matrices — default vs clinical threshold", fontsize=13)

for ax, thresh, title in zip(
    axes,
    [0.50, clinical_thresh],
    ["Default threshold (0.50)", f"Clinical threshold ({clinical_thresh:.2f}, recall≥0.80)"]
):
    pred_t = (calibrated_proba >= thresh).astype(int)
    cm     = confusion_matrix(y_test_fe, pred_t)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No CVD", "CVD"],
                yticklabels=["No CVD", "CVD"],
                linewidths=0.5, cbar=False)

    # Overlay percentages
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, f"({cm_pct[i,j]*100:.1f}%)",
                    ha="center", va="center", fontsize=9, color="grey")

    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{title}\nSensitivity={tp/(tp+fn):.3f} | Specificity={tn/(tn+fp):.3f} | "
                 f"F1={f1_score(y_test_fe, pred_t):.3f}",
                 fontsize=10)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")

plt.tight_layout()
plt.savefig("confusion_matrices.png", bbox_inches="tight")
plt.show()
print("Saved → confusion_matrices.png")

# ── 9. FINAL METRICS REPORT ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL METRICS — CALIBRATED ENSEMBLE (clinical threshold)")
print("=" * 60)
pred_clinical = (calibrated_proba >= clinical_thresh).astype(int)
print(classification_report(y_test_fe, pred_clinical,
                             target_names=["No CVD", "CVD"]))
print(f"ROC-AUC  : {roc_auc_score(y_test_fe, calibrated_proba):.4f}")
print(f"Brier    : {brier_score_loss(y_test_fe, calibrated_proba):.4f}")

# ── 10. SAVE DEPLOYMENT ARTIFACTS ─────────────────────────────────────────────
import joblib, json

joblib.dump(fe_fitted["XGBoost (FE)"],     "xgb_model.pkl")
joblib.dump(fe_fitted["LightGBM (FE)"],    "lgbm_model.pkl")
joblib.dump(preprocessor_fe,               "preprocessor.pkl")
joblib.dump(iso_cal,                        "calibrator.pkl")

# Save feature config for Gradio app
config = {
    "CONTINUOUS_FE":     CONTINUOUS_FE,
    "ORDINAL_FE":        ORDINAL_FE,
    "BINARY_FE":         BINARY_FE,
    "FEATURE_NAMES_FE":  FEATURE_NAMES_FE,
    "clinical_threshold": float(clinical_thresh),
    "best_f1_threshold":  float(best_f1_thresh),
}
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("\n── Saved deployment artifacts ──")
print("  xgb_model.pkl, lgbm_model.pkl, preprocessor.pkl,")
print("  calibrator.pkl, model_config.json")

print("\n✅ Snippet 4 complete.")
print("   Outputs: shap_beeswarm.png, shap_bar.png, shap_waterfall.png,")
print("            shap_dependence.png, calibration.png,")
print("            threshold_analysis.png, confusion_matrices.png")
print("   Proceed to Snippet 5 → Gradio app + HuggingFace Spaces deployment.")

Computing SHAP values (TreeExplainer)...
SHAP matrix shape: (13727, 20)

Plotting SHAP summary (beeswarm)...
Saved → shap_beeswarm.png
Saved → shap_bar.png
Saved → shap_waterfall.png
Saved → shap_dependence.png

Calibrating ensemble probabilities (isotonic regression)...
Saved → calibration.png

Threshold analysis:
  Default  (t=0.50) → F1=0.7179, Recall=0.6866
  Best F1  (t=0.36) → F1=0.7405, Recall=0.8522
  Clinical (t=0.20) → F1=0.7245, Recall=≥0.80
Saved → threshold_analysis.png
Saved → confusion_matrices.png

FINAL METRICS — CALIBRATED ENSEMBLE (clinical threshold)
              precision    recall  f1-score   support

      No CVD       0.83      0.40      0.54      6936
         CVD       0.60      0.92      0.72      6791

    accuracy                           0.66     13727
   macro avg       0.71      0.66      0.63     13727
weighted avg       0.72      0.66      0.63     13727

ROC-AUC  : 0.8009
Brier    : 0.1812

── Saved deployment artifacts ──
  xgb_model.pkl, lgbm_mode

In [ ]:
import subprocess
result = subprocess.run(
    ["pip", "install", "gradio==5.29.0", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-500:] if result.stderr else "")

import gradio as gr
print(f"Gradio version: {gr.__version__}")

In [10]:
# ============================================================
# SNIPPET 5 / 5 — GRADIO APP (KAGGLE-HOSTED) — POLISHED UI
# Cardiovascular Disease Classifier | Portfolio Project
# ============================================================

import subprocess
subprocess.run(["pip", "install", "gradio==5.29.0", "-q"])

import gradio as gr
import numpy as np
import pandas as pd
import joblib, json
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import shap, warnings
warnings.filterwarnings("ignore")

# ── 1. LOAD ARTIFACTS ─────────────────────────────────────────────────────────
xgb_model    = joblib.load("xgb_model.pkl")
lgbm_model   = joblib.load("lgbm_model.pkl")
preprocessor = joblib.load("preprocessor.pkl")
calibrator   = joblib.load("calibrator.pkl")

with open("model_config.json") as f:
    config = json.load(f)

CONTINUOUS_FE    = config["CONTINUOUS_FE"]
ORDINAL_FE       = config["ORDINAL_FE"]
BINARY_FE        = config["BINARY_FE"]
FEATURE_NAMES_FE = config["FEATURE_NAMES_FE"]
CLINICAL_THRESH  = config["clinical_threshold"]

explainer = shap.TreeExplainer(xgb_model)
print("Artifacts loaded ✓")

# ── 2. FEATURE ENGINEERING ────────────────────────────────────────────────────
AGE_MEAN  = df_fe["age_years"].mean()
AGE_STD   = df_fe["age_years"].std()
APHI_MEAN = df_fe["ap_hi"].mean()
APHI_STD  = df_fe["ap_hi"].std()

def jnc8_stage(ap_hi, ap_lo):
    if ap_hi >= 180 or ap_lo >= 120: return 4
    if ap_hi >= 140 or ap_lo >= 90:  return 3
    if ap_hi >= 130 or ap_lo >= 80:  return 2
    if ap_hi >= 120 and ap_lo < 80:  return 1
    return 0

def who_bmi_cat(bmi):
    if bmi < 18.5: return 0
    if bmi < 25.0: return 1
    if bmi < 30.0: return 2
    if bmi < 35.0: return 3
    return 4

def pp_cat(pp):
    if pp <= 40: return 0
    if pp <= 60: return 1
    return 2

def age_grp(age):
    if age < 40: return 0
    if age < 50: return 1
    if age < 60: return 2
    return 3

def build_feature_row(age_years, gender, height_cm, weight_kg,
                      ap_hi, ap_lo, cholesterol, gluc,
                      smoke, alco, active):
    bmi            = min(round(weight_kg / (height_cm / 100) ** 2, 2), 60)
    pulse_pressure = ap_hi - ap_lo
    map_val        = ap_lo + pulse_pressure / 3
    age_z          = (age_years - AGE_MEAN)  / AGE_STD
    aphi_z         = (ap_hi     - APHI_MEAN) / APHI_STD
    age_bp_inter   = age_z * aphi_z
    metabolic_risk   = int(cholesterol >= 2 and gluc >= 2)
    lifestyle_burden = int(smoke) + int(alco) + int(not active)
    htn_obese        = int(jnc8_stage(ap_hi, ap_lo) >= 2 and who_bmi_cat(bmi) >= 3)

    row = {
        "age_years": age_years, "bmi": bmi, "ap_hi": ap_hi,
        "ap_lo": ap_lo, "pulse_pressure": pulse_pressure,
        "map": map_val, "age_bp_interaction": age_bp_inter,
        "cholesterol": cholesterol, "gluc": gluc,
        "bp_stage": jnc8_stage(ap_hi, ap_lo),
        "bmi_category": who_bmi_cat(bmi),
        "pp_category": pp_cat(pulse_pressure),
        "age_group": age_grp(age_years),
        "gender": gender, "smoke": int(smoke), "alco": int(alco),
        "active": int(active), "metabolic_risk": metabolic_risk,
        "lifestyle_burden": lifestyle_burden, "htn_obese": htn_obese,
    }
    return pd.DataFrame([row])[FEATURE_NAMES_FE]

# ── 3. PREDICTION FUNCTION ────────────────────────────────────────────────────
def predict_cvd(age, gender_str, height, weight,
                ap_hi, ap_lo, chol_str, gluc_str,
                smoke, alco, active):

    gender = {"Female": 1, "Male": 2}[gender_str]
    chol   = {"Normal": 1, "Above normal": 2, "Well above normal": 3}[chol_str]
    gluc   = {"Normal": 1, "Above normal": 2, "Well above normal": 3}[gluc_str]

    X_raw  = build_feature_row(float(age), gender, float(height), float(weight),
                                float(ap_hi), float(ap_lo), chol, gluc,
                                smoke, alco, active)
    X_proc = preprocessor.transform(X_raw)

    p_raw  = (xgb_model.predict_proba(X_proc)[0,1] +
              lgbm_model.predict_proba(X_proc)[0,1]) / 2
    p_cal  = float(np.clip(calibrator.predict([p_raw])[0], 0.01, 0.99))

    # ── Risk tier ─────────────────────────────────────────────────────────────
    if p_cal >= 0.70:
        tier_icon  = "🔴"
        tier_label = "HIGH RISK"
        bar_color  = "#E74C3C"
        bg_color   = "#FDEDEC"
        advice     = "Strong indicators of cardiovascular disease present."
        action     = "Please consult a cardiologist as soon as possible."
    elif p_cal >= CLINICAL_THRESH:
        tier_icon  = "🟠"
        tier_label = "MODERATE RISK"
        bar_color  = "#E67E22"
        bg_color   = "#FEF9E7"
        advice     = "Elevated cardiovascular risk detected."
        action     = "Lifestyle changes and a medical review are recommended."
    else:
        tier_icon  = "🟢"
        tier_label = "LOW RISK"
        bar_color  = "#27AE60"
        bg_color   = "#EAFAF1"
        advice     = "Low cardiovascular risk based on current inputs."
        action     = "Maintain a healthy lifestyle and regular check-ups."

    # ── Result card HTML ──────────────────────────────────────────────────────
    bmi_val = min(round(float(weight) / (float(height)/100)**2, 1), 60)
    pp_val  = float(ap_hi) - float(ap_lo)

    result_html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; max-width: 520px;">

      <!-- Risk banner -->
      <div style="background:{bg_color}; border-left: 5px solid {bar_color};
                  border-radius:10px; padding:18px 20px; margin-bottom:14px;">
        <div style="font-size:28px; font-weight:700; color:{bar_color}; margin-bottom:4px;">
          {tier_icon} {tier_label}
        </div>
        <div style="font-size:42px; font-weight:800; color:{bar_color}; line-height:1;">
          {p_cal*100:.1f}%
        </div>
        <div style="font-size:12px; color:#666; margin-top:2px;">
          Predicted probability of cardiovascular disease
        </div>
      </div>

      <!-- Probability gauge -->
      <div style="background:#F0F0F0; border-radius:8px;
                  height:14px; margin-bottom:14px; overflow:hidden;">
        <div style="width:{p_cal*100:.1f}%; background:{bar_color};
                    height:100%; border-radius:8px; transition:width 0.4s;">
        </div>
      </div>

      <!-- Advice box -->
      <div style="background:#FAFAFA; border:1px solid #E0E0E0;
                  border-radius:10px; padding:14px 16px; margin-bottom:14px;">
        <div style="font-size:14px; color:#333; margin-bottom:4px;">
          💬 <strong>{advice}</strong>
        </div>
        <div style="font-size:13px; color:#555;">
          {action}
        </div>
      </div>

      <!-- Key vitals computed -->
      <div style="background:#F8F9FA; border:1px solid #E0E0E0;
                  border-radius:10px; padding:14px 16px; margin-bottom:14px;">
        <div style="font-size:12px; font-weight:600; color:#888;
                    text-transform:uppercase; margin-bottom:10px; letter-spacing:0.5px;">
          Computed Vitals
        </div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:8px;">
          <div style="background:white; border-radius:8px; padding:10px; text-align:center;
                      border:1px solid #EEE;">
            <div style="font-size:18px; font-weight:700; color:#2C3E50;">{bmi_val}</div>
            <div style="font-size:11px; color:#888;">BMI</div>
          </div>
          <div style="background:white; border-radius:8px; padding:10px; text-align:center;
                      border:1px solid #EEE;">
            <div style="font-size:18px; font-weight:700; color:#2C3E50;">{pp_val:.0f} mmHg</div>
            <div style="font-size:11px; color:#888;">Pulse Pressure</div>
          </div>
          <div style="background:white; border-radius:8px; padding:10px; text-align:center;
                      border:1px solid #EEE;">
            <div style="font-size:18px; font-weight:700; color:#2C3E50;">
              Stage {jnc8_stage(float(ap_hi), float(ap_lo))}
            </div>
            <div style="font-size:11px; color:#888;">BP Stage (JNC-8)</div>
          </div>
          <div style="background:white; border-radius:8px; padding:10px; text-align:center;
                      border:1px solid #EEE;">
            <div style="font-size:18px; font-weight:700; color:#2C3E50;">
              {"≥0.80" if p_cal >= CLINICAL_THRESH else "<0.80"}
            </div>
            <div style="font-size:11px; color:#888;">Clinical Threshold</div>
          </div>
        </div>
      </div>

      <!-- Footer -->
      <div style="font-size:11px; color:#AAA; text-align:center; padding-top:4px;">
        Model: Calibrated Ensemble (XGBoost + LightGBM) &nbsp;|&nbsp;
        ROC-AUC: 0.802 &nbsp;|&nbsp; n=68,634 patients<br>
        ⚠️ Educational tool only — not for clinical use.
      </div>
    </div>
    """

    # ── SHAP plot ─────────────────────────────────────────────────────────────
    shap_vals = explainer.shap_values(X_proc)[0]
    order     = np.argsort(np.abs(shap_vals))[-10:][::-1]
    feats     = [FEATURE_NAMES_FE[i] for i in order]
    vals      = shap_vals[order]
    raw_vals  = X_proc[0][order]

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor("#FAFAFA")
    ax.set_facecolor("#FAFAFA")

    bar_colors = ["#E74C3C" if v > 0 else "#3498DB" for v in vals[::-1]]
    bars = ax.barh(range(len(feats)), vals[::-1],
                   color=bar_colors, edgecolor="white",
                   linewidth=0.6, height=0.65)

    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats[::-1], fontsize=10)
    ax.axvline(0, color="#555", lw=1.0, linestyle="--")
    ax.set_xlabel("SHAP value  ←  lowers risk  |  raises risk  →", fontsize=10)
    ax.set_title(f"Why did the model predict {p_cal*100:.1f}% CVD risk?",
                 fontsize=11, fontweight="bold", pad=10)

    for bar, fv in zip(bars, raw_vals[::-1]):
        offset = 0.003 if bar.get_width() >= 0 else -0.003
        ha     = "left" if bar.get_width() >= 0 else "right"
        ax.text(bar.get_width() + offset,
                bar.get_y() + bar.get_height() / 2,
                f"{fv:.1f}", va="center", ha=ha, fontsize=8, color="#444")

    # Colour legend
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#E74C3C", label="Increases risk"),
                        Patch(color="#3498DB", label="Decreases risk")],
              fontsize=9, loc="lower right", framealpha=0.7)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()

    return result_html, fig

# ── 4. CUSTOM CSS ─────────────────────────────────────────────────────────────
css = """
.gradio-container { max-width: 1100px !important; margin: auto; }
.input-panel { background: #F8F9FA; border-radius: 12px; padding: 16px; }
.section-label {
    font-size: 11px; font-weight: 700; color: #888;
    text-transform: uppercase; letter-spacing: 0.6px;
    margin-bottom: 8px; margin-top: 14px;
}
footer { display: none !important; }
"""

# ── 5. GRADIO INTERFACE ───────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(
        primary_hue="blue",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Inter")
    ), css=css, title="CVD Risk Classifier") as demo:

    # Header
    gr.HTML("""
    <div style="text-align:center; padding: 24px 0 8px 0;">
      <div style="font-size:36px; margin-bottom:6px;">🫀</div>
      <div style="font-size:22px; font-weight:700; color:#1A1A2E;">
        Cardiovascular Disease Risk Classifier
      </div>
      <div style="font-size:13px; color:#666; margin-top:6px; max-width:560px; margin-inline:auto;">
        Enter patient vitals to estimate CVD risk using a calibrated ensemble model
        trained on <strong>68,634 patients</strong>.
        The model explains its reasoning using SHAP values.
      </div>
    </div>
    """)

    # Model badges
    gr.HTML("""
    <div style="display:flex; justify-content:center; gap:10px;
                flex-wrap:wrap; margin-bottom:20px;">
      <span style="background:#EBF5FB; color:#1A5276; font-size:11px;
                   padding:4px 12px; border-radius:20px; font-weight:600;">
        ROC-AUC: 0.802
      </span>
      <span style="background:#E9F7EF; color:#1E8449; font-size:11px;
                   padding:4px 12px; border-radius:20px; font-weight:600;">
        5-Fold Cross-Validated
      </span>
      <span style="background:#F4ECF7; color:#7D3C98; font-size:11px;
                   padding:4px 12px; border-radius:20px; font-weight:600;">
        XGBoost + LightGBM Ensemble
      </span>
      <span style="background:#FEF9E7; color:#B7770D; font-size:11px;
                   padding:4px 12px; border-radius:20px; font-weight:600;">
        Isotonic Calibration
      </span>
    </div>
    """)

    with gr.Row(equal_height=False):

        # ── LEFT: inputs ──────────────────────────────────────────────────────
        with gr.Column(scale=4, elem_classes="input-panel"):

            gr.HTML('<div class="section-label">👤 Patient Information</div>')
            with gr.Row():
                age    = gr.Slider(30, 65, value=50, step=1, label="Age (years)")
                gender = gr.Radio(["Female", "Male"], value="Female", label="Gender")
            with gr.Row():
                height = gr.Slider(140, 210, value=165, step=1, label="Height (cm)")
                weight = gr.Slider(40, 180, value=70,  step=1, label="Weight (kg)")

            gr.HTML('<div class="section-label">🩺 Clinical Measurements</div>')
            with gr.Row():
                ap_hi = gr.Slider(60,  250, value=120, step=1, label="Systolic BP (mmHg)")
                ap_lo = gr.Slider(40,  150, value=80,  step=1, label="Diastolic BP (mmHg)")
            with gr.Row():
                chol = gr.Dropdown(
                    ["Normal", "Above normal", "Well above normal"],
                    value="Normal", label="Cholesterol")
                gluc = gr.Dropdown(
                    ["Normal", "Above normal", "Well above normal"],
                    value="Normal", label="Glucose")

            gr.HTML('<div class="section-label">🏃 Lifestyle Factors</div>')
            with gr.Row():
                smoke  = gr.Checkbox(label="Smoker",             value=False)
                alco   = gr.Checkbox(label="Drinks alcohol",     value=False)
                active = gr.Checkbox(label="Physically active",  value=True)

            btn = gr.Button("🔍  Predict CVD Risk", variant="primary",
                            size="lg", scale=1)

        # ── RIGHT: outputs ────────────────────────────────────────────────────
        with gr.Column(scale=5):
            result_html = gr.HTML(
                value="""
                <div style="display:flex; align-items:center; justify-content:center;
                            height:180px; background:#F8F9FA; border-radius:12px;
                            border: 2px dashed #DDD; color:#AAA; font-size:14px;">
                  ← Fill in patient data and click Predict
                </div>
                """
            )
            shap_plot = gr.Plot(label="SHAP Feature Explanation",
                                show_label=True)

    # ── Examples ──────────────────────────────────────────────────────────────
    gr.HTML('<div style="margin-top:8px; margin-bottom:4px;" class="section-label">'
            '🧪 Example Patients — click any row to load</div>')
    gr.Examples(
        examples=[
            [58, "Male",   170, 95,  180, 110, "Well above normal", "Above normal",  True,  True,  False],
            [35, "Female", 165, 62,  110, 70,  "Normal",            "Normal",        False, False, True ],
            [52, "Male",   175, 88,  145, 92,  "Above normal",      "Normal",        True,  False, False],
            [44, "Female", 160, 75,  122, 81,  "Normal",            "Normal",        False, False, True ],
            [61, "Male",   168, 102, 165, 100, "Well above normal", "Above normal",  True,  True,  False],
        ],
        inputs=[age, gender, height, weight, ap_hi, ap_lo, chol, gluc,
                smoke, alco, active],
        label=" "
    )

    # Disclaimer
    gr.HTML("""
    <div style="text-align:center; font-size:11px; color:#BBB; margin-top:16px;">
      ⚠️ This tool is for <strong>educational and portfolio purposes only</strong>
      and does not constitute medical advice.
    </div>
    """)

    btn.click(
        fn=predict_cvd,
        inputs=[age, gender, height, weight, ap_hi, ap_lo,
                chol, gluc, smoke, alco, active],
        outputs=[result_html, shap_plot]
    )

# ── 6. LAUNCH ─────────────────────────────────────────────────────────────────
print("Launching app...")
demo.launch(share=True, debug=False, show_error=True, quiet=True)

Artifacts loaded ✓
Launching app...
* Running on public URL: https://922200a85b0d8d0c04.gradio.live
